# P-ML7 — Momentum Feature Engineering

**Motivation (F13):** P-ML5 RegimeEnsemble achieves Sharpe +0.927 but trails B&H (+1.379).
The primary identified weakness: **bull model Fold 2 IC = −0.050** (Jan 2021–Jan 2022,
the ATH + crash quarter). The current 12 features are dominated by single-bar oscillators
(`bar_ret`, `bb_zscore`, `rsi`) — none encodes whether price has been trending for weeks.

**Hypothesis (H1):** Adding multi-period momentum features (`ret_5`, `ret_20`, `ret_60`,
`mom_zscore_20`) gives the bull model an explicit trend-continuation signal, fixing Fold 2.

**Method:** IC screen on candidate features (stratified by regime) → select features with
positive bull IC → re-run P-ML5 `RegimeEnsemble` on augmented feature set → compare.

## §1 — Config

In [1]:
import sys
from pathlib import Path

repo_root = Path("__file__").resolve().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi":        120,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.size":         9,
})

# ── Dataset (identical to P-ML5) ──────────────────────────────────────────────
SYMBOL     = "BTC/USDT"
SINCE      = "2019-01-01"
UNTIL      = "2025-01-01"
HORIZON    = 1
N_SPLITS   = 5
TRAIN_FRAC = 0.6
PURGE      = 1
LONG_MA    = 200
ADX_THRESH = 25.0
MIN_BULL_BARS = 30

# ── Original 12-feature set (P-ML2 through P-ML6) ─────────────────────────────
FEATURES = [
    "bar_ret", "bb_zscore", "rsi", "macd_hist_norm", "atr_pct",
    "bb_width", "upper_wick", "lower_wick", "hl_range",
    "vol_log_chg", "di_diff", "adx",
]

# ── Momentum candidate features ───────────────────────────────────────────────
MOMENTUM_WINDOWS = (5, 20, 60)
ZSCORE_WINDOW    = 60     # ~3 months daily
IC_THRESHOLD     = 0.01   # minimum |IC| on bull bars to include a feature

# ── P-ML5 reference metrics (from F12) ───────────────────────────────────────
P_ML5_ICS    = [0.0612, -0.0536, 0.1295, 0.0462, 0.0861]   # fold-level ICs
P_ML5_BULL_ICS = [0.0312, -0.0497, float("nan"), 0.0421, 0.0611]  # bull-only ICs
BUY_HOLD     = {"return": 2.996, "sharpe": 1.379, "maxdd": -0.354}
P_ML5        = {"return": 6.302, "sharpe": 0.927, "maxdd": -0.680}
P_ML3_EXPC   = {"return": 0.088, "sharpe": 0.280, "maxdd": -0.498}

print(f"Dataset:          {SINCE} → {UNTIL} | 1d | horizon={HORIZON}")
print(f"Walk-forward:     {N_SPLITS} folds, train_frac={TRAIN_FRAC}, purge={PURGE}")
print(f"Momentum windows: {MOMENTUM_WINDOWS} bars | zscore_window={ZSCORE_WINDOW}")
print(f"IC threshold:     |IC_bull| > {IC_THRESHOLD} to include feature")
print(f"\nP-ML5 reference:  Sharpe={P_ML5['sharpe']:+.3f}, Return={P_ML5['return']*100:+.1f}%, MaxDD={P_ML5['maxdd']*100:.1f}%")

Dataset:          2019-01-01 → 2025-01-01 | 1d | horizon=1
Walk-forward:     5 folds, train_frac=0.6, purge=1
Momentum windows: (5, 20, 60) bars | zscore_window=60
IC threshold:     |IC_bull| > 0.01 to include feature

P-ML5 reference:  Sharpe=+0.927, Return=+630.2%, MaxDD=-68.0%


## §2 — Data Loading & Momentum Feature Construction

In [2]:
from data.fetch import fetch_ohlcv
from ml.features import build_feature_matrix
from ml.features.momentum import build_momentum_features
from ml.labels import forward_return
from ml.regime import RegimeClassifier
from ml.validation import purged_wf_splits

# Cache-guard (same as P-ML5)
df_raw = fetch_ohlcv(symbol=SYMBOL, timeframe="1d", since=SINCE, until=UNTIL)
if df_raw.index.min() > pd.Timestamp("2020-01-01", tz="UTC"):
    print("Cache missing early data — re-fetching 2019–2025 from exchange...")
    df_raw = fetch_ohlcv(symbol=SYMBOL, timeframe="1d",
                         since=SINCE, until=UNTIL, use_cache=False)
print(f"Loaded {len(df_raw):,} bars  |  {df_raw.index[0].date()} → {df_raw.index[-1].date()}")

# Build original features + momentum features + label
feats_orig = build_feature_matrix(df_raw)                           # original (no momentum)
feats_mom  = build_momentum_features(df_raw,
                                     windows=MOMENTUM_WINDOWS,
                                     zscore_window=ZSCORE_WINDOW)
label      = forward_return(df_raw, horizon=HORIZON)

# Combine and drop NaN (momentum has longest warmup: ret_60 needs 60 bars)
comb = pd.concat([feats_orig, feats_mom, label], axis=1).dropna()

X_orig = comb[FEATURES]
y_all  = comb[label.name]

# Momentum candidate columns
MOM_CANDIDATES = [f"ret_{w}" for w in MOMENTUM_WINDOWS] + ["mom_zscore_20", "ret_5_minus_20"]
X_mom = comb[MOM_CANDIDATES]

# Regime labels
rc  = RegimeClassifier(long_ma=LONG_MA, adx_thresh=ADX_THRESH)
reg = rc.transform(df_raw).reindex(comb.index)
reg["regime"] = reg["regime"].fillna("ranging")

bar_ret_daily = np.log(df_raw["close"] / df_raw["close"].shift(1)).reindex(comb.index)
splits = list(purged_wf_splits(len(comb), N_SPLITS, TRAIN_FRAC, purge_bars=PURGE))

print(f"\n{len(comb):,} usable bars | {comb.index[0].date()} → {comb.index[-1].date()}")
print(f"(vs P-ML5: same dataset — warmup only 60 bars for ret_60)")
print(f"\nMomentum feature sample (last 5 bars):")
print(X_mom.tail().to_string())

Cache missing early data — re-fetching 2019–2025 from exchange...


Loaded 2,193 bars  |  2019-01-01 → 2025-01-01

2,132 usable bars | 2019-03-02 → 2024-12-31
(vs P-ML5: same dataset — warmup only 60 bars for ret_60)

Momentum feature sample (last 5 bars):
                              ret_5    ret_20    ret_60  mom_zscore_20  ret_5_minus_20
timestamp                                                                             
2024-12-27 00:00:00+00:00 -0.009176 -0.056998  0.298584      -1.679366        0.047822
2024-12-28 00:00:00+00:00  0.004429 -0.059101  0.270112      -1.626621        0.063530
2024-12-29 00:00:00+00:00 -0.051193 -0.036962  0.259050      -1.384880       -0.014231
2024-12-30 00:00:00+00:00 -0.069150 -0.040100  0.277605      -1.366814       -0.029049
2024-12-31 00:00:00+00:00 -0.023217 -0.077411  0.297609      -1.615551        0.054194


## §3 — IC Screen: Momentum Features vs Forward Return

For each candidate momentum feature, compute Spearman IC against the 1d forward return
on: (a) all bars, (b) bull bars only, (c) non-bull bars only.

**Selection rule:** include feature if |IC_bull| > 0.01 (meaningful bull signal).
Also check collinearity with existing FEATURES (|r| > 0.8 = redundant).

In [3]:
bull_mask    = (reg["regime"] == "bull").values
nonbull_mask = ~bull_mask

print(f"Regime split: bull={bull_mask.sum()} bars ({bull_mask.mean()*100:.1f}%)  "
      f"non-bull={nonbull_mask.sum()} bars ({nonbull_mask.mean()*100:.1f}%)")
print()

# ── IC table ──────────────────────────────────────────────────────────────────
print(f"{'Feature':<18} {'IC_all':>8} {'p_all':>7} {'IC_bull':>9} {'p_bull':>8} "
      f"{'IC_nonbull':>11} {'p_nb':>7} {'Select?':>8}")
print("-" * 84)

ic_results = {}
for feat in MOM_CANDIDATES:
    x = X_mom[feat].values
    y = y_all.values

    rho_all,  p_all  = stats.spearmanr(x, y)
    rho_bull, p_bull = stats.spearmanr(x[bull_mask],    y[bull_mask])
    rho_nb,   p_nb   = stats.spearmanr(x[nonbull_mask], y[nonbull_mask])

    selected = abs(rho_bull) > IC_THRESHOLD
    ic_results[feat] = {
        "IC_all": rho_all, "p_all": p_all,
        "IC_bull": rho_bull, "p_bull": p_bull,
        "IC_nonbull": rho_nb, "p_nb": p_nb,
        "selected": selected,
    }
    sel_str = "YES ✓" if selected else "no"
    print(f"  {feat:<16} {rho_all:>+8.4f} {p_all:>7.3f} "
          f"{rho_bull:>+9.4f} {p_bull:>8.3f} "
          f"{rho_nb:>+11.4f} {p_nb:>7.3f} {sel_str:>8}")

selected_feats = [f for f, r in ic_results.items() if r["selected"]]
print(f"\nSelected momentum features ({len(selected_feats)}): {selected_feats}")

Regime split: bull=686 bars (32.2%)  non-bull=1446 bars (67.8%)

Feature              IC_all   p_all   IC_bull   p_bull  IC_nonbull    p_nb  Select?
------------------------------------------------------------------------------------
  ret_5             -0.0178   0.413   +0.0115    0.764     -0.0438   0.096    YES ✓
  ret_20            +0.0075   0.729   +0.0104    0.787     -0.0209   0.427    YES ✓
  ret_60            +0.0412   0.057   +0.0015    0.968     +0.0254   0.334       no
  mom_zscore_20     +0.0003   0.990   +0.0404    0.291     -0.0312   0.236    YES ✓
  ret_5_minus_20    +0.0004   0.986   -0.0123    0.748     +0.0264   0.316    YES ✓

Selected momentum features (4): ['ret_5', 'ret_20', 'mom_zscore_20', 'ret_5_minus_20']


In [4]:
# ── Collinearity check vs existing FEATURES ───────────────────────────────────
if selected_feats:
    corr_check = pd.concat([X_orig, X_mom[selected_feats]], axis=1).corr()
    cross_corr = corr_check.loc[selected_feats, FEATURES]
    print("Max |correlation| between selected momentum features and original 12 features:")
    for feat in selected_feats:
        max_corr = cross_corr.loc[feat].abs().max()
        max_feat = cross_corr.loc[feat].abs().idxmax()
        flag = " ← high overlap" if max_corr > 0.8 else ""
        print(f"  {feat:<20} max|r|={max_corr:.3f} with {max_feat}{flag}")

# ── IC bar chart: original features vs momentum features (bull bars) ──────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Original 12 features — bull IC
orig_bull_ics = {}
for feat in FEATURES:
    rho, _ = stats.spearmanr(X_orig[feat].values[bull_mask], y_all.values[bull_mask])
    orig_bull_ics[feat] = rho
orig_s = pd.Series(orig_bull_ics).sort_values()
colors_orig = ["#2ecc71" if v > 0 else "#e74c3c" for v in orig_s.values]
axes[0].barh(orig_s.index, orig_s.values, color=colors_orig, alpha=0.85)
axes[0].axvline(0, color="black", linewidth=0.7)
axes[0].set_title("Bull-bar IC: original 12 features", fontsize=9)
axes[0].set_xlabel("Spearman IC")

# Momentum candidates — bull IC
mom_bull_ics = {f: ic_results[f]["IC_bull"] for f in MOM_CANDIDATES}
mom_s = pd.Series(mom_bull_ics).sort_values()
colors_mom = ["#2ecc71" if v > 0 else "#e74c3c" for v in mom_s.values]
axes[1].barh(mom_s.index, mom_s.values, color=colors_mom, alpha=0.85)
axes[1].axvline(0, color="black", linewidth=0.7)
axes[1].axvline(IC_THRESHOLD,  color="gray", linewidth=0.8, linestyle="--",
                label=f"IC threshold ±{IC_THRESHOLD}")
axes[1].axvline(-IC_THRESHOLD, color="gray", linewidth=0.8, linestyle="--")
axes[1].set_title("Bull-bar IC: momentum candidates", fontsize=9)
axes[1].set_xlabel("Spearman IC")
axes[1].legend(fontsize=8)

plt.suptitle("Spearman IC on bull bars — original vs momentum features", fontsize=10)
plt.tight_layout()
plt.show()

Max |correlation| between selected momentum features and original 12 features:
  ret_5                max|r|=0.748 with bb_zscore
  ret_20               max|r|=0.848 with rsi ← high overlap
  mom_zscore_20        max|r|=0.721 with macd_hist_norm
  ret_5_minus_20       max|r|=0.618 with rsi


## §4 — Augmented Walk-Forward with RegimeEnsemble

Re-run the P-ML5 walk-forward using `FEATURES_V2 = FEATURES + selected_feats`.
Primary focus: does bull model Fold 2 IC (Jan 2021–Jan 2022) turn positive?

In [5]:
from ml.models import RegimeEnsemble
from backtesting import compute_metrics

# Build augmented feature set
FEATURES_V2 = FEATURES + selected_feats
X_v2 = comb[FEATURES_V2]
print(f"FEATURES_V2 ({len(FEATURES_V2)} features): {FEATURES_V2}")
print()

# ── Walk-forward loop ─────────────────────────────────────────────────────────
fold_results_P7 = []
bull_ics_p7 = []
nb_ics_p7   = []

print(f"{'Fold':<5} {'Period':<35} {'n_train':>8} {'bull_tr':>7} "
      f"{'P5 IC':>8} {'P7 IC':>8} {'Δ IC':>7} {'hit':>6}")
print("-" * 88)

for i, (tr, te) in enumerate(splits):
    ensemble = RegimeEnsemble(min_bull_bars=MIN_BULL_BARS)
    ensemble.fit(X_v2.iloc[tr], y_all.iloc[tr], reg["regime"].iloc[tr])
    preds  = ensemble.predict(X_v2.iloc[te], reg["regime"].iloc[te])
    actual = y_all.iloc[te].values

    rho, pval = stats.spearmanr(preds, actual)
    hit       = (np.sign(preds) == np.sign(actual)).mean()

    # Bull IC on bull test bars
    bull_te = (reg["regime"].iloc[te] == "bull").values
    nb_te   = ~bull_te
    if bull_te.sum() >= 2:
        if ensemble.has_bull_model:
            pb = ensemble.bull_model.predict(X_v2.iloc[te][bull_te])
        else:
            pb = -ensemble.non_bull_model.predict(X_v2.iloc[te][bull_te])
        bull_ic_p7, _ = stats.spearmanr(pb, actual[bull_te])
    else:
        bull_ic_p7 = float("nan")
    bull_ics_p7.append(bull_ic_p7)

    if nb_te.sum() >= 2:
        pnb = ensemble.non_bull_model.predict(X_v2.iloc[te][nb_te])
        nb_ic_p7, _ = stats.spearmanr(pnb, actual[nb_te])
    else:
        nb_ic_p7 = float("nan")
    nb_ics_p7.append(nb_ic_p7)

    n_bull_tr = int((reg["regime"].iloc[tr] == "bull").sum())
    p5_ic     = P_ML5_ICS[i]
    delta     = rho - p5_ic
    period    = f"{comb.index[te[0]].date()} → {comb.index[te[-1]].date()}"

    print(f"  {i+1:<3} {period:<35} {len(tr):>8} {n_bull_tr:>7} "
          f"{p5_ic:>+8.4f} {rho:>+8.4f} {delta:>+7.4f} {hit:>6.3f}")

    fold_results_P7.append({
        "fold": i + 1, "te": te, "IC": rho, "IC_pval": pval,
        "hit": hit, "preds": preds, "ensemble": ensemble,
    })

ics_P7 = [r["IC"] for r in fold_results_P7]
print()
print(f"P-ML7: Mean IC={np.mean(ics_P7):+.4f}  ICIR={np.mean(ics_P7)/np.std(ics_P7):.3f}")
print(f"P-ML5: Mean IC={np.mean(P_ML5_ICS):+.4f}  ICIR={np.mean(P_ML5_ICS)/np.std(P_ML5_ICS):.3f}  [reference]")

FEATURES_V2 (16 features): ['bar_ret', 'bb_zscore', 'rsi', 'macd_hist_norm', 'atr_pct', 'bb_width', 'upper_wick', 'lower_wick', 'hl_range', 'vol_log_chg', 'di_diff', 'adx', 'ret_5', 'ret_20', 'mom_zscore_20', 'ret_5_minus_20']

Fold  Period                               n_train bull_tr    P5 IC    P7 IC    Δ IC    hit
----------------------------------------------------------------------------------------
  1   2020-02-20 → 2021-02-08                  354      37  +0.0612  +0.0721 +0.0109  0.459
  2   2021-02-09 → 2022-01-29                  531     216  -0.0536  +0.0091 +0.0627  0.513


  3   2022-01-30 → 2023-01-19                  531     253  +0.1295  +0.1283 -0.0012  0.569
  4   2023-01-20 → 2024-01-09                  531      92  +0.0462  +0.0537 +0.0075  0.521


  5   2024-01-10 → 2024-12-29                  531     214  +0.0861  +0.1069 +0.0208  0.569

P-ML7: Mean IC=+0.0740  ICIR=1.779
P-ML5: Mean IC=+0.0539  ICIR=0.888  [reference]


In [6]:
# ── Bull model IC comparison: P-ML5 vs P-ML7 ─────────────────────────────────
print(f"{'Fold':<5} {'Period':<35} {'Bull IC P5':>11} {'Bull IC P7':>11} "
      f"{'Δ Bull IC':>10} {'Improved?':>10}")
print("-" * 85)

for i, (r, bic_p7) in enumerate(zip(fold_results_P7, bull_ics_p7)):
    _, te = splits[i]
    period  = f"{comb.index[te[0]].date()} → {comb.index[te[-1]].date()}"
    bic_p5  = P_ML5_BULL_ICS[i]
    delta   = bic_p7 - bic_p5 if not np.isnan(bic_p7) and not np.isnan(bic_p5) else float("nan")
    improved = "YES" if not np.isnan(delta) and delta > 0 else ("—" if np.isnan(delta) else "no")
    bic_p5_str = f"{bic_p5:+.4f}" if not np.isnan(bic_p5) else "  nan "
    bic_p7_str = f"{bic_p7:+.4f}" if not np.isnan(bic_p7) else "  nan "
    delta_str  = f"{delta:+.4f}" if not np.isnan(delta) else "   nan"
    print(f"  {r['fold']:<3} {period:<35} {bic_p5_str:>11} {bic_p7_str:>11} "
          f"{delta_str:>10} {improved:>10}")

valid_p5 = [ic for ic in P_ML5_BULL_ICS if not np.isnan(ic)]
valid_p7 = [ic for ic in bull_ics_p7     if not np.isnan(ic)]
print(f"\nBull IC — P-ML5 Mean={np.nanmean(P_ML5_BULL_ICS):+.4f}  P-ML7 Mean={np.nanmean(bull_ics_p7):+.4f}")
print(f"Non-bull IC — P-ML5 Mean={np.nanmean([0.1576,-0.0531,0.1295,0.0631,0.0892]):+.4f}  "
      f"P-ML7 Mean={np.nanmean(nb_ics_p7):+.4f}")

Fold  Period                               Bull IC P5  Bull IC P7  Δ Bull IC  Improved?
-------------------------------------------------------------------------------------
  1   2020-02-20 → 2021-02-08                 +0.0312        nan         nan          —
  2   2021-02-09 → 2022-01-29                 -0.0497     -0.1278    -0.0781         no
  3   2022-01-30 → 2023-01-19                    nan      +0.1786        nan          —
  4   2023-01-20 → 2024-01-09                 +0.0421     +0.0582    +0.0161        YES
  5   2024-01-10 → 2024-12-29                 +0.0611     +0.0714    +0.0103        YES

Bull IC — P-ML5 Mean=+0.0212  P-ML7 Mean=+0.0451
Non-bull IC — P-ML5 Mean=+0.0773  P-ML7 Mean=+0.0742


## §5 — Equity Comparison & Finding F14

In [7]:
def build_equity(fold_preds_list, bar_ret_daily, idx):
    pieces, anchor = [], 1.0
    for te, preds in fold_preds_list:
        pos   = np.sign(preds)
        ret   = bar_ret_daily.iloc[te].values
        eq    = np.cumprod(1 + np.roll(pos, 1) * ret)
        eq[0] = 1.0
        s = pd.Series(eq, index=idx[te])
        s = s / s.iloc[0] * anchor
        anchor = float(s.iloc[-1])
        pieces.append(s)
    return pd.concat(pieces)


idx = comb.index
oos_P7 = build_equity([(r["te"], r["preds"]) for r in fold_results_P7], bar_ret_daily, idx)
bah    = df_raw["close"].reindex(oos_P7.index)
bah    = bah / bah.iloc[0]

m_p7  = compute_metrics(oos_P7)

# ── Equity plot ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
oos_P7.plot(ax=ax, label="P-ML7 (LGBM + momentum features)", color="#9b59b6", linewidth=2.0)
bah.plot(   ax=ax, label="Buy-and-Hold", color="black", linewidth=1.0, linestyle=":")
for _, (tr, te) in enumerate(splits):
    ax.axvline(idx[te[0]], color="lightgray", linewidth=0.5, linestyle="--")
ax.axhline(1, color="black", linewidth=0.4)
ax.set_title("OOS Equity — P-ML7 (momentum features) vs Buy-and-Hold", fontsize=11)
ax.set_ylabel("Equity (normalised)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# ── Fold-by-fold IC chart: P-ML5 vs P-ML7 ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(N_SPLITS)
w = 0.35
ax.bar(x - w/2, P_ML5_ICS, w, label="P-ML5 LightGBM (12 features)",
       color="#3498db", alpha=0.85)
ax.bar(x + w/2, ics_P7,    w, label=f"P-ML7 LightGBM ({len(FEATURES_V2)} features)",
       color="#9b59b6", alpha=0.85)
ax.axhline(0,     color="black", linewidth=0.7)
ax.axhline(+0.03, color="gray",  linewidth=0.6, linestyle="--", label="±0.03 target")
ax.axhline(-0.03, color="gray",  linewidth=0.6, linestyle="--")
ax.set_xticks(x)
ax.set_xticklabels([f"F{i+1}" for i in range(N_SPLITS)])
ax.set_ylabel("OOS IC")
ax.set_title("OOS IC per fold: P-ML5 (12 features) vs P-ML7 (+momentum)", fontsize=10)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# ── Metrics table ─────────────────────────────────────────────────────────────
print(f"\n{'Strategy':<35} {'Return':>9} {'Sharpe':>8} {'MaxDD':>8}")
print("-" * 63)
rows = [
    ("Buy & Hold",             BUY_HOLD["return"],   BUY_HOLD["sharpe"],   BUY_HOLD["maxdd"]),
    ("P-ML3 Exp-C (best 3yr)", P_ML3_EXPC["return"], P_ML3_EXPC["sharpe"], P_ML3_EXPC["maxdd"]),
    ("P-ML5 (6yr, 12 feats)",  P_ML5["return"],      P_ML5["sharpe"],      P_ML5["maxdd"]),
    (f"P-ML7 (6yr, {len(FEATURES_V2)} feats)",
                               m_p7["total_return"],  m_p7["sharpe_ratio"],  m_p7["max_drawdown"]),
]
for name, ret, shr, mdd in rows:
    print(f"  {name:<33} {ret*100:>+8.1f}%  {shr:>+7.3f}  {mdd*100:>7.1f}%")


Strategy                               Return   Sharpe    MaxDD
---------------------------------------------------------------
  Buy & Hold                          +299.6%   +1.379    -35.4%
  P-ML3 Exp-C (best 3yr)                +8.8%   +0.280    -49.8%
  P-ML5 (6yr, 12 feats)               +630.2%   +0.927    -68.0%
  P-ML7 (6yr, 16 feats)              +1997.6%   +1.261    -77.3%


In [8]:
# ── Finding F14 summary ───────────────────────────────────────────────────────
p7_return = m_p7["total_return"]
p7_sharpe = m_p7["sharpe_ratio"]
p7_maxdd  = m_p7["max_drawdown"]

mean_bull_ic_p5 = np.nanmean(P_ML5_BULL_ICS)
mean_bull_ic_p7 = np.nanmean(bull_ics_p7)

fold2_bull_p5 = P_ML5_BULL_ICS[1]   # Fold 2 = index 1
fold2_bull_p7 = bull_ics_p7[1]

ic_improved      = np.mean(ics_P7) > np.mean(P_ML5_ICS)
bull_ic_improved = mean_bull_ic_p7 > mean_bull_ic_p5
fold2_fixed      = (not np.isnan(fold2_bull_p7)) and (fold2_bull_p7 > 0)
sharpe_improved  = p7_sharpe > P_ML5["sharpe"]

print("=" * 70)
print("FINDING F14 — Momentum Features (P-ML7) vs LightGBM Baseline (P-ML5)")
print("=" * 70)
print()
print(f"Selected momentum features: {selected_feats}")
print(f"Feature set: {len(FEATURES)} → {len(FEATURES_V2)} features")
print()
print(f"{'Metric':<35} {'P-ML5 (12f)':>13} {'P-ML7 (+mom)':>14}")
print("-" * 65)
print(f"  {'Mean OOS IC':<33} {np.mean(P_ML5_ICS):>+13.4f} {np.mean(ics_P7):>+14.4f}")
print(f"  {'ICIR':<33} {np.mean(P_ML5_ICS)/np.std(P_ML5_ICS):>+13.3f} "
      f"{np.mean(ics_P7)/np.std(ics_P7):>+14.3f}")
print(f"  {'Mean Bull IC':<33} {mean_bull_ic_p5:>+13.4f} {mean_bull_ic_p7:>+14.4f}")
print(f"  {'Fold 2 Bull IC (ATH+crash)':<33} {fold2_bull_p5:>+13.4f} "
      f"{fold2_bull_p7:>+14.4f}")
print(f"  {'OOS Sharpe':<33} {P_ML5['sharpe']:>+13.3f} {p7_sharpe:>+14.3f}")
print(f"  {'OOS Return':<33} {P_ML5['return']*100:>+12.1f}% {p7_return*100:>+13.1f}%")
print(f"  {'Max Drawdown':<33} {P_ML5['maxdd']*100:>+12.1f}% {p7_maxdd*100:>+13.1f}%")
print()
print(f"Overall IC improved?       {'YES' if ic_improved      else 'NO'}")
print(f"Mean bull IC improved?     {'YES' if bull_ic_improved  else 'NO'}")
print(f"Fold 2 bull IC positive?   {'YES' if fold2_fixed       else 'NO'} "
      f"(was {fold2_bull_p5:+.4f}, now {fold2_bull_p7:+.4f})")
print(f"Sharpe improved?           {'YES' if sharpe_improved   else 'NO'}")
print()

if sharpe_improved and bull_ic_improved:
    verdict = (
        "HYPOTHESIS SUPPORTED: Momentum features improve both bull model IC and overall "
        "Sharpe. Multi-period returns provide trend-continuation signal that the "
        "single-bar oscillators cannot encode."
    )
elif bull_ic_improved or ic_improved:
    verdict = (
        "HYPOTHESIS PARTIALLY SUPPORTED: Momentum features improve IC but not equity "
        "(or vice-versa). The additional signal is real but the translation to profit "
        "may be limited by the position sizing (binary +1/-1) or regime gating."
    )
else:
    verdict = (
        "HYPOTHESIS REJECTED: Momentum features do not improve IC or equity vs P-ML5. "
        "Multi-period returns may already be captured implicitly by MACD and DI signals, "
        "or the bull Fold 2 failure has a different root cause. "
        "Next step: H2 — longer forecast horizon (3d/5d) to escape mean-reversion dominance."
    )

print(f"VERDICT: {verdict}")
print()
print("Next steps:")
if not sharpe_improved:
    print("  - H2: Try HORIZON=3 or HORIZON=5 — longer targets reduce mean-reversion noise")
    print("  - H3: HMM regime classifier — detect regime transitions earlier than SMA200+ADX")
if sharpe_improved:
    print("  - P-ML8: Wrap RegimeEnsemble into MLStrategy class (strategy integration)")
    print("  - P-ML9: Risk overlay — position sizing + drawdown brake to reduce MaxDD")
    print("  - P-ML11: Optuna hyperparameter tuning on augmented feature set")

FINDING F14 — Momentum Features (P-ML7) vs LightGBM Baseline (P-ML5)

Selected momentum features: ['ret_5', 'ret_20', 'mom_zscore_20', 'ret_5_minus_20']
Feature set: 12 → 16 features

Metric                                P-ML5 (12f)   P-ML7 (+mom)
-----------------------------------------------------------------
  Mean OOS IC                             +0.0539        +0.0740
  ICIR                                     +0.888         +1.779
  Mean Bull IC                            +0.0212        +0.0451
  Fold 2 Bull IC (ATH+crash)              -0.0497        -0.1278
  OOS Sharpe                               +0.927         +1.261
  OOS Return                              +630.2%       +1997.6%
  Max Drawdown                             -68.0%         -77.3%

Overall IC improved?       YES
Mean bull IC improved?     YES
Fold 2 bull IC positive?   NO (was -0.0497, now -0.1278)
Sharpe improved?           YES

VERDICT: HYPOTHESIS SUPPORTED: Momentum features improve both bull model IC an